In [7]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = [
    'raw',
    'filling_anatomy_gaps'
]


In [8]:
import numpy as np
import matplotlib.pyplot as plt
from core.file_manager import preprocess_file_manager
from core.anatomy_gap_fixer import find_gaps_in_anatomy

In [9]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)
patients = file_manager.get_file_names()

checking size

In [10]:
biggest_gap = (0, -1, 0)
smallest_gap = (0,99999, 0)
_, sections_with_prostate = find_gaps_in_anatomy(patients,file_manager) 
for patient in sections_with_prostate:
    diff = sections_with_prostate[patient][1] - sections_with_prostate[patient][0] + 1
    if diff > biggest_gap[1]:
        biggest_gap = (patient,diff,sections_with_prostate[patient])
    if diff < smallest_gap[1]:
        smallest_gap = (patient,diff,sections_with_prostate[patient])

print(f"biggest prostate height {biggest_gap}")
print(f"smallest prostate height {smallest_gap}")

biggest prostate height ('3322', 16, (15, 30))
smallest prostate height ('3322', 16, (15, 30))


looking for smallest picture

In [11]:
def find_smallest_dimensions_of_3D_arrays(patients):
    smallest_dim = [9999 for x in range (0,3)]

    for patient in patients:
        data = file_manager.load_file('raw', patient)
        shape_per_channel = [data[channel].shape for channel in channels]

        all_same = len(set(shape_per_channel)) == 1
        if not all_same:
            print(f"ALERT ALERT {patient}")

        for i in range(0,3):
            dim = shape_per_channel[0][i]
            if smallest_dim[i] > dim:
                smallest_dim[i] = dim
    return smallest_dim

smallest_dim = find_smallest_dimensions_of_3D_arrays(patients)
for i in range(0,3):
    print(f"smallest dim{i} {smallest_dim[i]}")

smallest dim0 320
smallest dim1 320
smallest dim2 41


In [12]:
from stat_calc import find_centroid_mean, find_centroid_non_weighted

prostate = file_manager.load_file('raw', patient)['anatomy']

np_centroid = find_centroid_mean(prostate)
print("Np Centroid:", np_centroid)

non_weighted_centroid = find_centroid_non_weighted(prostate)
print("Np Centroid:", non_weighted_centroid)

Np Centroid: [160.9385006  163.20979779  21.95169232]
Np Centroid: [160.0, 161.0, 22.5]
